In [1]:
import ee 
import geemap
ee.Authenticate()
ee.Initialize()

# Panama boundary
panama = (
    ee.FeatureCollection('FAO/GAUL/2015/level0')
    .filter(ee.Filter.eq('ADM0_NAME', 'Panama'))
)

# Hansen dataset (clipped)
dataset = ee.Image('UMD/hansen/global_forest_change_2025_v1_13').clip(panama)

lossyear = dataset.select('lossyear')

# DEFINE "RECENT LOSS"
# Example: last 5 years in dataset timeline
# Hansen lossyear is typically:
# 1 = 2001, ..., 25 = 2025
recent_threshold = 20  # ~2020 onward (adjust if needed)

recent_loss = lossyear.gte(recent_threshold).selfMask()

# DISTANCE TO RECENT LOSS
distance_recent = (
    recent_loss
    .fastDistanceTransform(256)
    .sqrt()
    .multiply(30)   # meters
    .rename('dist_recent_loss_m')
)

# VISUALIZATION
vis = {
    'min': 0,
    'max': 10000,  # 10 km range works well
    'palette': ['red', 'orange', 'yellow', 'green', 'blue']
}

# MAP
m = geemap.Map(center=[8.5, -80], zoom=7)

m.add_layer(
    recent_loss,
    {'palette': ['ff0000']},
    'Recent forest loss (binary)'
)

m.add_layer(
    distance_recent,
    vis,
    'Distance to recent forest loss (m)'
)

m.add_layer(panama, {}, 'Panama boundary')

m

Map(center=[8.5, -80], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tr…